# M4 single test evaluation (GPU)

**What this is.** The one run of the frozen pipeline on the official FinQA test split (1,147 records),
following [docs/m4-test-protocol.md](../docs/m4-test-protocol.md) exactly: the retrained Notebook 10
retriever, top 10 facts, the Notebook 17 Qwen2.5-1.5B LoRA adapter, the top of 4 beams as the answer,
and the share of 16 samples agreeing with it as confidence (answer when ≥ 1.0; secondary ≥ 0.75).

**Order.** Sections 1–3 build predictions from allowlisted inputs only (`id`, `question`, `table`,
`pre_text`, `post_text`) and write them to disk. Only section 4 reads test labels, to score. Per-record
timings are kept as the inference record. Needs a CUDA GPU (Kaggle T4); `FINEVIDENCE_SMOKE_TEST=1`
runs a tiny CPU check on 6 records with local smoke models.


In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import os
import platform
import random
import subprocess
import time
import tomllib
from pathlib import Path

import numpy as np
import pandas as pd
import peft
import torch
import transformers
from peft import PeftModel
from scipy.stats import binomtest
from transformers import AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer

from finevidence.data import load_finqa_split
from finevidence.data.acquisition import download_finqa_splits
from finevidence.evaluation import gold_recall_at_k
from finevidence.evaluation.abstention import coverage_curve
from finevidence.generation import (
    build_retrieved_examples,
    score_program_prediction,
    summarize_scores,
)
from finevidence.generation.causal import INSTRUCTION, PROGRAM_MARKER, extract_program
from finevidence.reasoning.executor import execute_program
from finevidence.retrieval import build_ranking_pairs, load_retriever_config, rank_keys

PROJECT_ROOT = (
    Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
)
SMOKE_TEST = os.environ.get("FINEVIDENCE_SMOKE_TEST") == "1"
ADAPTER_SHA256 = "1646a66b99ae80c1b738420d94b6d589af0b1796b03c9d8cd915ea433a58b4f5"
PRIMARY_THRESHOLD, SECONDARY_THRESHOLD = 1.0, 0.75
cfg = tomllib.loads((PROJECT_ROOT / "configs" / "generator_qwen_lora_top10.toml").read_text())
retriever_cfg = load_retriever_config(PROJECT_ROOT / "configs" / "retriever_bert_base.toml")
pretrained = (
    "trl-internal-testing/tiny-Qwen2ForCausalLM-2.5"
    if SMOKE_TEST
    else cfg["model"]["pretrained_name"]
)
num_beams = 2 if SMOKE_TEST else cfg["generation"]["num_beams"]
num_samples = 4 if SMOKE_TEST else cfg["sampling"]["num_samples"]
facts = cfg["training"]["facts_per_input"]
suffix = "-smoke" if SMOKE_TEST else ""
output_dir = PROJECT_ROOT / "data" / "processed"
KAGGLE_INPUT = Path("/kaggle/input")


def find_input_dir(marker: str, accept=lambda directory: True) -> Path | None:
    """First directory under /kaggle/input holding `marker` that passes `accept`."""
    if not KAGGLE_INPUT.exists():
        return None
    for path in sorted(KAGGLE_INPUT.rglob(marker)):
        if accept(path.parent):
            return path.parent
    return None


def is_bert_classifier(directory: Path) -> bool:
    config_file = directory / "config.json"
    return (directory / "model.safetensors").exists() and "BertForSequenceClassification" in (
        config_file.read_text()
    )


retriever_dir = Path(
    os.environ.get("FINEVIDENCE_RETRIEVER_DIR")
    or find_input_dir("config.json", is_bert_classifier)
    or PROJECT_ROOT / retriever_cfg.model_dir / "best"
)
adapter_dir = Path(
    os.environ.get("FINEVIDENCE_ADAPTER_DIR")
    or find_input_dir("adapter_model.safetensors")
    or PROJECT_ROOT / cfg["output"]["model_dir"] / "best"
)
adapter_sha = hashlib.sha256((adapter_dir / "adapter_model.safetensors").read_bytes()).hexdigest()
if not SMOKE_TEST:
    assert adapter_sha == ADAPTER_SHA256, adapter_sha

seed = cfg["seed"]
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if not SMOKE_TEST and device.type != "cuda":
    raise RuntimeError(
        "The test run needs a CUDA GPU; set FINEVIDENCE_SMOKE_TEST=1 for a CPU check."
    )
use_fp16 = device.type == "cuda"
commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], capture_output=True, text=True, cwd=PROJECT_ROOT
).stdout.strip()
environment = {
    "commit": commit,
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "peft": peft.__version__,
    "device": torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu",
    "smoke_test": SMOKE_TEST,
    "retriever_dir": str(retriever_dir),
    "adapter_dir": str(adapter_dir),
    "adapter_sha256": adapter_sha,
    "pretrained": pretrained,
    "facts": facts,
    "num_beams": num_beams,
    "num_samples": num_samples,
    "seed": seed,
}
print(json.dumps(environment, indent=2))

## 1. Test split (verified) and retrieval

The retriever scores every segment of each test record; nothing here reads labels.


In [ ]:
raw_dir = PROJECT_ROOT / "data" / "raw"
print(download_finqa_splits(raw_dir, ("test",)))
test = load_finqa_split(raw_dir / "test.json")
if SMOKE_TEST:
    test = test[:6]
inputs = [
    {
        key: record[key]
        for key in ("id", "question", "table", "pre_text", "post_text")
        if key in record
    }
    | {"qa": {"question": record["qa"]["question"]}}
    for record in test
]


def clock(seconds: float) -> str:
    minutes, seconds = divmod(int(seconds), 60)
    hours, minutes = divmod(minutes, 60)
    return f"{hours}h{minutes:02d}m{seconds:02d}s" if hours else f"{minutes}m{seconds:02d}s"


retriever_tokenizer = AutoTokenizer.from_pretrained(retriever_dir)
retriever = AutoModelForSequenceClassification.from_pretrained(retriever_dir).to(device).eval()
rankings, retrieval_seconds = {}, {}
started = time.time()
with torch.no_grad():
    for record in inputs:
        record_started = time.time()
        pairs = build_ranking_pairs(record, include_header_row=retriever_cfg.include_header_row)
        scores = []
        for start in range(0, len(pairs), retriever_cfg.batch_size * 4):
            batch = pairs[start : start + retriever_cfg.batch_size * 4]
            encoded = retriever_tokenizer(
                [pair.question for pair in batch],
                [pair.text for pair in batch],
                truncation="only_second",
                max_length=retriever_cfg.max_seq_length,
                padding=True,
                return_tensors="pt",
            ).to(device)
            with torch.autocast(device.type, enabled=use_fp16):
                logits = retriever(**encoded).logits
            scores.extend(torch.softmax(logits.float(), dim=-1)[:, 1].tolist())
        rankings[record["id"]] = rank_keys([pair.key for pair in pairs], scores)
        retrieval_seconds[record["id"]] = time.time() - record_started
print(f"ranked {len(rankings)} test records in {clock(time.time() - started)}")
rankings_path = output_dir / f"retriever_bert_base{suffix}_test_rankings.jsonl"
rankings_path.parent.mkdir(parents=True, exist_ok=True)
with rankings_path.open("w", encoding="utf-8") as handle:
    for record_id, ranked in rankings.items():
        handle.write(json.dumps({"id": record_id, "ranked_keys": ranked, "commit": commit}) + "\n")
del retriever
gc.collect()
torch.cuda.empty_cache()

## 2. Generator: beams and samples

Same decoding code and settings as Notebook 17.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(adapter_dir)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
max_source_tokens = cfg["model"]["max_source_tokens"]
marker_ids = tokenizer(PROGRAM_MARKER, add_special_tokens=False).input_ids


def prompt_ids(example) -> list[int]:
    source = tokenizer(INSTRUCTION + example.source, add_special_tokens=False).input_ids
    return [*source[:max_source_tokens], *marker_ids]


examples = build_retrieved_examples(inputs, rankings, facts_per_input=facts)
model = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(
        pretrained, torch_dtype=torch.float16 if use_fp16 else torch.float32
    ),
    adapter_dir,
).to(device)
model.config.use_cache = False


@torch.no_grad()
def decode(
    examples, beams: int, samples: int = 0, batch_size: int | None = None
) -> tuple[list[list[str]], list[list[float]]]:
    """Beam search (samples=0) or sampling; returns programs and length-normalized log-probs."""
    model.eval()
    model.config.use_cache = True
    tokenizer.padding_side = "left"
    outputs, all_scores = [], []
    width_per = samples or beams
    batch_size = batch_size or cfg["generation"]["eval_batch_size"]
    for start in range(0, len(examples), batch_size):
        batch = examples[start : start + batch_size]
        prompts = [prompt_ids(example) for example in batch]
        width = max(len(p) for p in prompts)
        input_ids = torch.tensor(
            [[tokenizer.pad_token_id] * (width - len(p)) + p for p in prompts], device=device
        )
        attention = torch.tensor(
            [[0] * (width - len(p)) + [1] * len(p) for p in prompts], device=device
        )
        options = (
            {
                "do_sample": True,
                "num_beams": 1,
                "temperature": cfg["sampling"]["temperature"],
                "top_p": cfg["sampling"]["top_p"],
            }
            if samples
            else {"do_sample": False, "num_beams": beams}
        )
        with torch.autocast(device.type, dtype=torch.float16, enabled=use_fp16):
            generated = model.generate(
                input_ids=input_ids,
                attention_mask=attention,
                max_new_tokens=cfg["model"]["max_new_tokens"],
                num_return_sequences=width_per,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                output_scores=True,
                return_dict_in_generate=True,
                **options,
            )
        new_tokens = generated.sequences[:, width:]
        if beams > 1 and not samples:
            # Beam search already returns the length-normalized log-probability of each sequence.
            sequence_scores = generated.sequences_scores.float().tolist()
        else:
            transition = model.compute_transition_scores(
                generated.sequences, generated.scores, normalize_logits=True
            )
            valid = new_tokens[:, : transition.shape[1]] != tokenizer.pad_token_id
            lengths = valid.sum(dim=1).clamp(min=1)
            sequence_scores = (
                transition.float().masked_fill(~valid, 0).sum(dim=1) / lengths
            ).tolist()
        texts = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
        programs = [extract_program(text) for text in texts]
        for i in range(0, len(programs), width_per):
            outputs.append(programs[i : i + width_per])
            all_scores.append([round(x, 5) for x in sequence_scores[i : i + width_per]])
    model.config.use_cache = False
    return outputs, all_scores


started = time.time()
beam_candidates, beam_scores = decode(examples, beams=num_beams)
beam_seconds = time.time() - started
print(f"decoded {len(beam_candidates)} records with {num_beams} beams in {clock(beam_seconds)}")
torch.manual_seed(seed)
started = time.time()
sample_candidates, sample_scores = decode(
    examples, beams=1, samples=num_samples, batch_size=cfg["sampling"]["sample_batch_size"]
)
sample_seconds = time.time() - started
print(f"sampled {num_samples} programs per record in {clock(sample_seconds)}")

## 3. Answers and confidence, written before scoring


In [ ]:
answers, confidences = [], []
for record, beams, samples in zip(inputs, beam_candidates, sample_candidates, strict=True):
    answer = beams[0]
    value = execute_program(answer, record["table"])
    votes = [execute_program(program, record["table"]) for program in samples]
    agreeing = sum(v.is_valid and v.value == value.value for v in votes) if value.is_valid else 0
    answers.append(answer)
    confidences.append(agreeing / len(samples))
predictions_path = output_dir / f"m4-test{suffix}_predictions.jsonl"
with predictions_path.open("w", encoding="utf-8") as handle:
    for example, answer, confidence, beams, bscores, samples, sscores in zip(
        examples,
        answers,
        confidences,
        beam_candidates,
        beam_scores,
        sample_candidates,
        sample_scores,
        strict=True,
    ):
        row = {
            "record_id": example.record_id,
            "fact_keys": example.fact_keys,
            "answer": answer,
            "confidence": confidence,
            "beam_candidates": beams,
            "beam_scores": bscores,
            "sample_candidates": samples,
            "sample_scores": sscores,
            "commit": commit,
        }
        handle.write(json.dumps(row) + "\n")
records = len(inputs)
inference_record = {
    "records": records,
    "retrieval_seconds_per_record_mean": round(float(np.mean(list(retrieval_seconds.values()))), 3),
    "beam_seconds_per_record": round(beam_seconds / records, 3),
    "sample_seconds_per_record": round(sample_seconds / records, 3),
    "total_seconds_per_record": round(
        float(np.mean(list(retrieval_seconds.values())))
        + (beam_seconds + sample_seconds) / records,
        3,
    ),
    "device": environment["device"],
}
(output_dir / f"m4-test{suffix}_inference_record.json").write_text(
    json.dumps({"environment": environment, **inference_record}, indent=2)
)
print(predictions_path.relative_to(PROJECT_ROOT), inference_record)

## 4. Scoring (first read of test labels)


In [ ]:
def interval(k: int, n: int) -> str:
    if n == 0:
        return "n/a"
    ci = binomtest(k, n).proportion_ci()
    return f"{k}/{n} = {k / n:.2%} (95% CI {ci.low:.1%}-{ci.high:.1%})"


scores = [
    score_program_prediction(record, answer, example.fact_texts)
    for record, answer, example in zip(test, answers, examples, strict=True)
]
summary = summarize_scores(scores)
correct = [score.execution_correct for score in scores]
recall = {
    k: float(
        np.mean(
            [
                gold_recall_at_k(rankings[r["id"]], list(r["qa"]["gold_inds"]), k).all_gold
                for r in test
                if r["qa"]["gold_inds"]
            ]
        )
    )
    for k in (5, 10)
}
selective = {}
for name, threshold in (("primary", PRIMARY_THRESHOLD), ("secondary", SECONDARY_THRESHOLD)):
    answered = [ok for ok, c in zip(correct, confidences, strict=True) if c >= threshold]
    selective[name] = {
        "threshold": threshold,
        "coverage": interval(len(answered), len(correct)),
        "accuracy": interval(sum(answered), len(answered)),
    }
report = {
    "execution_accuracy": interval(sum(correct), len(correct)),
    "program_accuracy": interval(sum(s.program_correct for s in scores), len(scores)),
    **{f"summary_{k}": v for k, v in summary.items()},
    "all_gold_recall_at_5": round(recall[5], 4),
    "all_gold_recall_at_10": round(recall[10], 4),
    "selective": selective,
}
(output_dir / f"m4-test{suffix}_report.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))
display(pd.DataFrame([vars(p) for p in coverage_curve(confidences, correct)]))